# VGGT Multi-Input Robustness Experiments

**Central question:** How robust is the pretrained VGGT model when reconstructing geometry and camera relationships from different real-world and non-ideal multi-view image sets?

This is the main seminar notebook. It supports manifest-backed custom captures and the approved ETH3D `delivery_area` / `courtyard` subset. It never downloads data, and all inference remains behind an explicit approval gate. ETH3D ground truth is inventoried here but quantitative evaluation is future work.


## 1. Setup and offline imports
Select the `vggt-seminar` kernel. Offline flags are set before VGGT imports; root discovery contains no user-specific path.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import gc, json, platform, sys, time
from pathlib import Path

def discover_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate/'pyproject.toml').is_file() and (candidate/'configs/default.yaml').is_file(): return candidate
    raise FileNotFoundError(f"Could not find project root from {current}")

ROOT = discover_root()
for path in (ROOT/'src', ROOT/'external/vggt'):
    if str(path) not in sys.path: sys.path.insert(0, str(path))

import numpy as np
import torch
import yaml
from IPython.display import HTML, display
from PIL import Image, ImageDraw
from vggt_seminar.eth3d import (apply_order as eth3d_apply_order, build_experiment_configurations,
    discover_scenes as discover_eth3d_scenes, load_dataset_manifest as load_eth3d_manifest,
    load_scene as load_eth3d_scene, scene_summary, select_frames as select_eth3d_frames)
from vggt_seminar.eth3d_overlap import PROTOCOL_VERSION, load_frozen_selection
from vggt_seminar.experiments import (confidence_summary, degrade_image, input_manifest,
    inventory_scenes, load_scene_manifest, normalized_point_disagreement,
    ordered_scene_images, ordered_variant, select_evenly, statuses_as_dicts)
from vggt_seminar.live_demo import (center_or_explicit_query, heatmap_rgb, point_cloud_preview,
    prediction_schema, save_heatmap, save_ply, verify_local_assets)
from vggt.models.vggt import VGGT
from vggt.utils.geometry import unproject_depth_map_to_point_map
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
SEED=42; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); np.random.seed(SEED)
print(f"Project root: {ROOT}")


## 2. Evidence model

- **Paper reproduction:** official pretrained architecture/checkpoint and output definitions only.
- **Our experiment:** controlled local inputs, frame subsets, order, and degradations.
- **Qualitative evidence:** coherence, artifacts, camera layout, and confidence/failure correspondence.
- **Quantitative diagnostics:** runtime, VRAM, finiteness, confidence summaries, and point-branch disagreement.
- **Not supported without ground truth:** metric depth/pose accuracy or exact paper reproduction.

## 3. Inventory available input sources

Custom scenes require a complete `scene_manifest.yaml`. ETH3D scenes are discovered only when their images and calibration are present under the ignored local dataset root.


In [ ]:
CUSTOM_INPUTS = ROOT / "data/custom_inputs"
ETH3D_ROOT = ROOT / "local_assets/datasets/eth3d"

scene_statuses = inventory_scenes(CUSTOM_INPUTS)
rows = statuses_as_dicts(scene_statuses)
display(HTML("<h4>Custom scenes</h4><table><tr><th>Scene</th><th>Images</th><th>Manifest</th><th>Ready</th><th>Issue</th></tr>" +
    "".join(f"<tr><td>{r['name']}</td><td>{r['image_count']}</td><td>{r['manifest_exists']}</td><td>{r['valid']}</td><td>{r['issue'] or ''}</td></tr>" for r in rows) + "</table>"))

eth3d_names = discover_eth3d_scenes(ETH3D_ROOT)
eth3d_dataset_manifest = load_eth3d_manifest(ETH3D_ROOT)
display(HTML("<h4>ETH3D scenes</h4><pre>" + json.dumps(eth3d_names, indent=2) + "</pre>"))


## 4. Editable experiment configuration
The safety flags default to false. Setting `BUILD_FULL_PLAN=True` only constructs a plan; inference still requires `APPROVE_INFERENCE=True`.

In [ ]:
INPUT_SOURCE = "eth3d"  # options: eth3d, custom
SCENE_NAME = "delivery_area"
FRAME_COUNTS = [2, 4, 6, 8, 10]
SELECTION_STRATEGIES = ["overlap_aware_nested"]
ORDER_VARIANTS = ["original", "reversed", "shuffled"]
SELECTED_FRAME_COUNT = 2
SELECTED_SELECTION_STRATEGY = "overlap_aware_nested"
SELECTED_ORDER = "original"
DEGRADATION_MODES = ["none"]  # options: none, blur, low_light, jpeg, low_resolution
DEGRADATION_SEVERITY = 0.5
RUN_FULL_RESOLUTION = True  # official preprocessing still targets 518 px
ENABLE_TRACKING = True
TRACK_QUERY_XY = None
EXPORT_RESULTS = True
RUN_NAME = "phase6_1_overlap_aware_preview"
BUILD_FULL_PLAN = False
APPROVE_INFERENCE = False
DEVICE = "cuda"
USE_BFLOAT16 = True
PREPROCESSING_MODE = "crop"
assert INPUT_SOURCE in {"eth3d", "custom"}
assert DEVICE == "cuda", "No silent CPU fallback is permitted."


## 5. Load the selected input scene

ETH3D uses official COLMAP calibration order. Custom input uses its required scene manifest. There is no cartoon or synthetic fallback.


In [ ]:
if INPUT_SOURCE == "eth3d":
    eth3d_scene = load_eth3d_scene(ETH3D_ROOT, SCENE_NAME)
    all_image_paths = list(eth3d_scene.image_paths)
    manifest = {"source": "eth3d", **scene_summary(eth3d_scene)}
else:
    scene_dir = CUSTOM_INPUTS / SCENE_NAME
    manifest = load_scene_manifest(scene_dir)
    all_image_paths = ordered_scene_images(scene_dir, manifest)

if len(all_image_paths) < 2:
    raise ValueError("The selected scene requires at least two views.")
display(HTML("<pre>" + json.dumps(manifest, indent=2, default=str) + "</pre>"))
print(f"Validated {len(all_image_paths)} ordered images from {INPUT_SOURCE}:{SCENE_NAME}.")


OVERLAP_CONFIG = ROOT / "configs/experiments/phase6_1_overlap_aware_frames.yaml"
frozen_selections = {}
if INPUT_SOURCE == "eth3d" and "overlap_aware_nested" in SELECTION_STRATEGIES:
    for count in FRAME_COUNTS:
        frozen_selections[count] = load_frozen_selection(OVERLAP_CONFIG, SCENE_NAME, count)
        actual = [all_image_paths[i].name for i in frozen_selections[count]["indices"]]
        if actual != frozen_selections[count]["filenames"]:
            raise ValueError(f"Frozen S{count} filenames do not match the local calibrated scene")
    print(f"Frozen protocol: {PROTOCOL_VERSION}")
    display(HTML("<pre>" + json.dumps(frozen_selections, indent=2) + "</pre>"))


## 6. Inspect a contact sheet before inference

Confirm scene identity, sequence, overlap, dimensions, and calibration availability before approving any model execution.


In [ ]:
thumb_size = (240, 160)
if INPUT_SOURCE == "eth3d" and SELECTED_SELECTION_STRATEGY == "overlap_aware_nested":
    frozen_preview = frozen_selections[SELECTED_FRAME_COUNT]
    shown = [all_image_paths[index] for index in frozen_preview["indices"]]
    print("Nested sets:", {f"S{count}": frozen_selections[count]["indices"] for count in FRAME_COUNTS})
    print("Selected filenames:", frozen_preview["filenames"])
    print("Overlap diagnostics:", frozen_preview["diagnostics"])
else:
    shown = all_image_paths[:min(12, len(all_image_paths))]
columns = 4
rows_count = (len(shown) + columns - 1) // columns
sheet = Image.new("RGB", (columns * thumb_size[0], rows_count * (thumb_size[1] + 24)), "white")
draw = ImageDraw.Draw(sheet)
for index, path in enumerate(shown):
    with Image.open(path) as source:
        image = source.convert("RGB")
        image.thumbnail(thumb_size)
    x = (index % columns) * thumb_size[0]
    y = (index // columns) * (thumb_size[1] + 24)
    sheet.paste(image, (x, y))
    draw.text((x + 4, y + thumb_size[1] + 3), f"{index:02d} {path.name}", fill="black")
display(sheet)


## 7. Construct the controlled run plan
Frozen overlap-aware subsets are loaded strictly from the Phase 6.1 config. Missing scenes/counts fail closed; there is no selection fallback. Order remains an explicit later experiment factor.


In [ ]:
if INPUT_SOURCE == "eth3d":
    full_plan = build_experiment_configurations(
        len(all_image_paths), FRAME_COUNTS, SELECTION_STRATEGIES, ORDER_VARIANTS, SEED
    )
else:
    full_plan = [
        {"frame_count": count, "selection_strategy": strategy, "order": order, "seed": SEED}
        for count in FRAME_COUNTS if count <= len(all_image_paths)
        for strategy in SELECTION_STRATEGIES for order in ORDER_VARIANTS
    ]

if BUILD_FULL_PLAN:
    run_plan = [{**condition, "degradation": degradation} for condition in full_plan for degradation in DEGRADATION_MODES]
else:
    if SELECTED_FRAME_COUNT > len(all_image_paths):
        raise ValueError("SELECTED_FRAME_COUNT exceeds available images.")
    run_plan = [{
        "frame_count": SELECTED_FRAME_COUNT,
        "selection_strategy": SELECTED_SELECTION_STRATEGY,
        "order": SELECTED_ORDER,
        "seed": SEED,
        "degradation": DEGRADATION_MODES[0],
    }]
print(f"Planned {len(run_plan)} condition(s); this does not execute them.")
display(HTML("<pre>" + json.dumps(run_plan, indent=2) + "</pre>"))


## 8. Verify CUDA, code pin, and local checkpoint
The full checkpoint SHA-256 is checked before model allocation. This may take a little while.

In [ ]:
if not torch.cuda.is_available(): raise RuntimeError("CUDA unavailable; refusing CPU fallback.")
assets = verify_local_assets(ROOT)
gpu = torch.cuda.get_device_properties(0)
print({"python":platform.python_version(),"torch":torch.__version__,"cuda":torch.version.cuda,
       "gpu":torch.cuda.get_device_name(0),"compute_capability":torch.cuda.get_device_capability(0),
       "vram_gib":round(gpu.total_memory/2**30,2),"vggt_commit":assets['pin']['commit'],
       "checkpoint_verified":assets['checkpoint_hash_matches']})

## 9. Explicit execution gate
Review the run plan. Change `APPROVE_INFERENCE=True` only when you intentionally want to run the selected conditions.

In [ ]:
if not APPROVE_INFERENCE:
    raise RuntimeError("Inference is not approved. Review the plan, then set APPROVE_INFERENCE=True in the configuration cell.")
print("Inference explicitly approved for the displayed run plan.")

## 10. Initialize and load the official model once
The checkpoint is loaded locally in offline mode and the model is moved to CUDA once for all approved conditions.

In [ ]:
model_start=time.perf_counter(); model=VGGT(); architecture_init_seconds=time.perf_counter()-model_start
load_start=time.perf_counter(); state=torch.load(assets['checkpoint'],map_location='cpu',weights_only=True,mmap=True)
model.load_state_dict(state,strict=True); del state; checkpoint_load_seconds=time.perf_counter()-load_start
device=torch.device('cuda:0'); torch.cuda.set_device(0)
transfer_start=time.perf_counter(); model=model.to(device).eval(); torch.cuda.synchronize(); transfer_seconds=time.perf_counter()-transfer_start
print(f"Initialized={architecture_init_seconds:.3f}s, checkpoint={checkpoint_load_seconds:.3f}s, transfer={transfer_seconds:.3f}s")

## 11. Condition runner
This wrapper uses official preprocessing and one model forward per condition. Derived degradation images are written under that condition's output directory, never over originals.

In [ ]:
def run_condition(condition):
    if INPUT_SOURCE == 'eth3d':
        if condition['selection_strategy'] == 'overlap_aware_nested':
            frozen = load_frozen_selection(OVERLAP_CONFIG, SCENE_NAME, condition['frame_count'])
            selected = [all_image_paths[index] for index in frozen['indices']]
            if [path.name for path in selected] != frozen['filenames']:
                raise ValueError('Frozen filenames do not match local calibrated order')
        else:
            selected=select_eth3d_frames(all_image_paths,condition['frame_count'],condition['selection_strategy'])
        selected=eth3d_apply_order(selected,condition['order'],seed=condition['seed'])
    else:
        selected=list(all_image_paths[:condition['frame_count']]) if condition['selection_strategy']=='sequential' else select_evenly(all_image_paths,condition['frame_count'])
        selected=ordered_variant(selected,condition['order'],seed=condition['seed'])
    condition_id=f"{INPUT_SOURCE}_{SCENE_NAME}_n{condition['frame_count']}_{condition['selection_strategy']}_{condition['order']}_{condition['degradation']}"
    condition_dir=ROOT/'outputs/predictions'/RUN_NAME/condition_id
    if condition_dir.exists(): raise FileExistsError(f"Refusing to overwrite {condition_dir}")
    raw_dir=condition_dir/'raw'; vis_dir=condition_dir/'visualizations'; derived_dir=condition_dir/'derived_inputs'
    raw_dir.mkdir(parents=True); vis_dir.mkdir()
    inference_paths=selected
    if condition['degradation']!='none':
        derived_dir.mkdir(); inference_paths=[]
        for source in selected:
            target=derived_dir/source.name
            degrade_image(Image.open(source),condition['degradation'],DEGRADATION_SEVERITY).save(target)
            inference_paths.append(target)
    images_cpu=load_and_preprocess_images(inference_paths,mode=PREPROCESSING_MODE)
    images=images_cpu.to(device); height,width=images.shape[-2:]
    query=center_or_explicit_query(width,height,TRACK_QUERY_XY,device) if ENABLE_TRACKING else None
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0); baseline=torch.cuda.memory_allocated(0)
    started=time.perf_counter()
    with torch.inference_mode(), torch.autocast(device_type='cuda',dtype=torch.bfloat16 if USE_BFLOAT16 else torch.float16):
        predictions=model(images,query_points=query)
    torch.cuda.synchronize(); elapsed=time.perf_counter()-started; peak=torch.cuda.max_memory_allocated(0)
    extrinsic,intrinsic=pose_encoding_to_extri_intri(predictions['pose_enc'].float(),images.shape[-2:])
    unprojected=unproject_depth_map_to_point_map(predictions['depth'].float().squeeze(0).cpu().numpy(),extrinsic.squeeze(0).cpu().numpy(),intrinsic.squeeze(0).cpu().numpy())
    direct=predictions['world_points'].float().cpu().numpy()[0]
    diagnostics=normalized_point_disagreement(direct,unprojected)
    diagnostics['depth_confidence']=confidence_summary(predictions['depth_conf'])
    diagnostics['point_confidence']=confidence_summary(predictions['world_points_conf'])
    runtime={"inference_seconds":elapsed,"baseline_gpu_bytes":int(baseline),"peak_gpu_bytes":int(peak),"frame_count":len(selected)}
    schema=prediction_schema(predictions,{"extrinsic":extrinsic,"intrinsic":intrinsic,"unprojected_points":torch.from_numpy(unprojected)})
    if EXPORT_RESULTS:
        for name,value in predictions.items():
            if isinstance(value,torch.Tensor): torch.save(value.detach().cpu(),raw_dir/f"{name}.pt")
            elif isinstance(value,list):
                for index,item in enumerate(value): torch.save(item.detach().cpu(),raw_dir/f"{name}_{index}.pt")
        for name,value in {"extrinsic":extrinsic,"intrinsic":intrinsic,"unprojected_points":torch.from_numpy(unprojected)}.items(): torch.save(value.detach().cpu(),raw_dir/f"{name}.pt")
        (condition_dir/'condition.json').write_text(json.dumps(condition,indent=2)+'\n',encoding='utf-8')
        (condition_dir/'runtime.json').write_text(json.dumps(runtime,indent=2)+'\n',encoding='utf-8')
        (condition_dir/'diagnostics.json').write_text(json.dumps(diagnostics,indent=2)+'\n',encoding='utf-8')
        (condition_dir/'output_schema.json').write_text(json.dumps(schema,indent=2)+'\n',encoding='utf-8')
        (condition_dir/'input_manifest.json').write_text(json.dumps(input_manifest(selected,ROOT),indent=2)+'\n',encoding='utf-8')
        save_heatmap(predictions['depth'][0,0],vis_dir/'depth_view0.png',invert=True)
        save_heatmap(predictions['depth_conf'][0,0],vis_dir/'depth_conf_view0.png')
        save_heatmap(predictions['world_points_conf'][0,0],vis_dir/'point_conf_view0.png')
        colors=predictions['images'][0].permute(0,2,3,1).float().cpu().numpy()
        save_ply(direct,colors,vis_dir/'direct_points.ply'); save_ply(unprojected,colors,vis_dir/'unprojected_points.ply')
    return {"condition_id":condition_id,"condition":condition,"runtime":runtime,"diagnostics":diagnostics,"schema":schema,"predictions":predictions,"extrinsic":extrinsic,"intrinsic":intrinsic,"direct":direct,"unprojected":unprojected}
print("Condition runner defined; nothing has run yet.")


## 12. Execute only the approved plan
For the first study, keep this to one or a few deliberate conditions. Do not turn the complete taxonomy into an automatic sweep.

In [ ]:
results=[]
for condition in run_plan:
    print(f"Running {condition}")
    results.append(run_condition(condition))
print(f"Completed {len(results)} approved condition(s).")

## 13. Quantitative diagnostic table
These values compare conditions internally; they are not benchmark accuracy metrics.

In [ ]:
summary=[]
for result in results:
    d=result['diagnostics']; r=result['runtime']
    summary.append({"condition":result['condition_id'],"views":r['frame_count'],"seconds":r['inference_seconds'],"peak_GiB":r['peak_gpu_bytes']/2**30,"point_disagreement":d['median_distance_normalized'],"depth_conf_median":d['depth_confidence']['median'],"point_conf_median":d['point_confidence']['median']})
display(HTML("<table><tr>"+"".join(f"<th>{k}</th>" for k in summary[0])+"</tr>"+"".join("<tr>"+"".join(f"<td>{v:.4f}</td>" if isinstance(v,float) else f"<td>{v}</td>" for v in row.values())+"</tr>" for row in summary)+"</table>"))

## 14. Qualitative inspection figures
Inspect depth, confidence, direct geometry, and depth-unprojected geometry together. Look for floaters, broken planes, foreground/background leakage, reflections, and moving-object duplication.

In [ ]:
for result in results:
    predictions=result['predictions']; colors=predictions['images'][0].permute(0,2,3,1).float().cpu().numpy()
    display(HTML(f"<h3>{result['condition_id']}</h3><h4>Depth / depth confidence / point confidence (view 0)</h4>"))
    display(Image.fromarray(heatmap_rgb(predictions['depth'][0,0],invert=True)),Image.fromarray(heatmap_rgb(predictions['depth_conf'][0,0])),Image.fromarray(heatmap_rgb(predictions['world_points_conf'][0,0])))
    display(HTML("<h4>Direct / depth-unprojected point previews</h4>"),point_cloud_preview(result['direct'],colors),point_cloud_preview(result['unprojected'],colors))

## 15. Camera and order stability
Camera poses are expressed relative to the first frame and have gauge/scale ambiguity. Compare relative relationships after explicit alignment; do not compare raw translations across changed reference frames as absolute error.

In [ ]:
for result in results:
    ext=result['extrinsic'].detach().cpu().numpy()[0]; intr=result['intrinsic'].detach().cpu().numpy()[0]
    print(result['condition_id'], 'camera count=',len(ext))
    print('focal estimates:',intr[:,0,0].round(3).tolist())
    print('camera-from-world translations:',ext[:,:3,3].round(4).tolist())

## 16. Record qualitative observations
Replace placeholders with concrete observations. Separate visible evidence from interpretation.

In [ ]:
observations={
    "geometry_coherence": "TODO: planes, object shape, floaters, discontinuities",
    "camera_plausibility": "TODO: path/order and obvious flips",
    "direct_vs_unprojected": "TODO: visible and diagnostic differences",
    "confidence_alignment": "TODO: where confidence agrees/disagrees with visible failures",
    "strengths": [], "failure_modes": [],
    "conclusion_scope": "Qualitative/diagnostic only unless ground truth is documented",
}
display(HTML('<pre>'+yaml.safe_dump(observations,sort_keys=False)+'</pre>'))

## 17. Fine-tuning decision gate

Do not fine-tune from this notebook. First require a repeated failure, a held-out metric, legal data, a module-specific hypothesis, a simpler baseline, and an approved one-batch memory probe. The current most defensible candidate—only if evidence supports it—is frozen-aggregator adaptation of a small depth or camera head. See `docs/phase4_finetuning_feasibility.md`.

## 18. Cleanup
Run after analysis/export. CPU result dictionaries remain unless explicitly removed.

In [ ]:
for name in ['model']:
    globals().pop(name,None)
for result in globals().get('results',[]): result.pop('predictions',None)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"CUDA allocated={torch.cuda.memory_allocated(0)/2**20:.1f} MiB, reserved={torch.cuda.memory_reserved(0)/2**20:.1f} MiB")

## 19. Phase 7 saved-results review (no inference)
This section reads the canonical aggregate produced by the approved batch runner. It does not load the checkpoint or execute VGGT. Geometry plots use arbitrary, unaligned VGGT scale.


In [ ]:
import csv
PHASE7_ROOT = ROOT / 'outputs/experiments/phase7_delivery_area_view_count'
SUMMARY_CSV = PHASE7_ROOT / 'summary.csv'
if not SUMMARY_CSV.is_file():
    raise FileNotFoundError(f'Run the approved Phase 7 batch runner first: {SUMMARY_CSV}')
with SUMMARY_CSV.open(encoding='utf-8', newline='') as handle:
    phase7_rows = list(csv.DictReader(handle))
headers = list(phase7_rows[0])
table = '<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>'
for row in phase7_rows:
    table += '<tr>' + ''.join(f'<td>{row[h]}</td>' for h in headers) + '</tr>'
display(HTML(table + '</table>'))
for plot in sorted((PHASE7_ROOT / 'plots').glob('*.png')):
    display(HTML(f'<h4>{plot.stem}</h4>')); display(Image.open(plot))
for panel in sorted((PHASE7_ROOT / 'comparisons').glob('*.jpg')):
    display(HTML(f'<h4>{panel.stem}</h4>')); display(Image.open(panel))
for row in phase7_rows:
    subset = row['subset']; artifact = ROOT / ('outputs/predictions/phase6_2_eth3d_overlap_smoke/delivery_area/S2_overlap_aware_nested_original' if subset == 'S2' else f'outputs/predictions/phase7_eth3d_view_count/delivery_area/{subset}_overlap_aware_nested_original')
    print(f'{subset}: {artifact}')


## 20. Phase 8 courtyard and two-scene saved-results review (no inference)
This section reads Phase 7/8 canonical aggregates and comparison panels. It never loads VGGT or executes inference; camera/geometry values remain arbitrary and unaligned.


In [ ]:
PHASE8_ROOT = ROOT / 'outputs/experiments/phase8_courtyard_view_count'
COURTYARD_CSV = PHASE8_ROOT / 'summary.csv'
DELIVERY_CSV = ROOT / 'outputs/experiments/phase7_delivery_area_view_count/summary.csv'
CROSS_CSV = PHASE8_ROOT / 'comparisons/delivery_area_vs_courtyard.csv'
for required in (COURTYARD_CSV, DELIVERY_CSV, CROSS_CSV):
    if not required.is_file(): raise FileNotFoundError(required)
def display_saved_csv(path):
    with path.open(encoding='utf-8', newline='') as handle: rows=list(csv.DictReader(handle))
    headers=list(rows[0]); html='<table><tr>'+''.join(f'<th>{h}</th>' for h in headers)+'</tr>'
    for row in rows: html+='<tr>'+''.join(f'<td>{row[h]}</td>' for h in headers)+'</tr>'
    display(HTML(f'<h3>{path.name}</h3>'+html+'</table>')); return rows
courtyard_rows=display_saved_csv(COURTYARD_CSV)
delivery_rows=display_saved_csv(DELIVERY_CSV)
cross_scene_rows=display_saved_csv(CROSS_CSV)
for folder in ('plots','comparisons','contact_sheets','point_cloud_previews'):
    for figure in sorted((PHASE8_ROOT/folder).glob('*.png'))+sorted((PHASE8_ROOT/folder).glob('*.jpg')):
        display(HTML(f'<h4>{folder}/{figure.name}</h4>')); display(Image.open(figure))


## 21. Phase 9 report synthesis (saved artifacts only)

This final section reads the Phase 9 report dataset, tables, and figures. It performs no checkpoint loading or inference. The editable report is at [`../report/vggt_seminar_report.docx`](../report/vggt_seminar_report.docx), with the Markdown source at [`../report/vggt_seminar_report.md`](../report/vggt_seminar_report.md).


In [ ]:
import csv
from IPython.display import Image, Markdown, display

REPORT_ROOT = ROOT / 'report'
RESULTS_CSV = REPORT_ROOT / 'data/view_count_results.csv'
if not RESULTS_CSV.is_file():
    raise FileNotFoundError(f'Generate saved-only report artifacts first: {RESULTS_CSV}')
with RESULTS_CSV.open(encoding='utf-8', newline='') as handle:
    report_rows = list(csv.DictReader(handle))
display(Markdown(f'**Unified report dataset:** {len(report_rows)} configurations'))
for table_name in ('table03_delivery_runtime.md', 'table04_courtyard_runtime.md', 'table07_cross_scene.md'):
    display(Markdown((REPORT_ROOT / 'tables' / table_name).read_text(encoding='utf-8')))
for figure_name in ('fig01_total_time.png', 'fig02_vram.png', 'fig03_depth_confidence.png', 'fig06_contact_sheets.png'):
    display(Image(filename=str(REPORT_ROOT / 'figures' / figure_name)))
display(Markdown('[Open the editable seminar report](../report/vggt_seminar_report.docx)'))
